# Hotel-Analytics mit pandas

Unsere Hoteldatenbank ist voll. In den nächsten rund 60 Minuten schauen wir mit **pandas** hinein: wenige Zeilen Code, viele Plots, keine Statistikformeln.

| Block | Thema | Zeit |
|---|---|---|
| 1 | Wer sind unsere Gäste? | 10 Min. |
| 2 | Kalender-Detektive | 10 Min. |
| 3 | Preisfalle und Umsatzkurve | 20 Min. |
| 4 | SpiegelMining im Hotel | 15 Min. |
| 5 | Darf man das? | 5 Min. |

**Wichtig:** Die Termine von Events und Feiertagen sind real, ihre Effekte auf Preise und Buchungen sind synthetisch erzeugt. Alle Personen, Firmen und Buchungen sind frei erfunden.

Zellen mit einem Kommentar `# Hinweis:` füllt ihr selbst aus.

## 1. Wer sind unsere Gäste? (10 Min.)

In den Daten stecken Privat- und Geschäftsreisende, aber keine Spalte verrät, wer wer ist. Zuerst laden wir die Daten.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
DATA = Path("../data/export")
cities = pd.read_parquet(DATA / "cities.parquet")
hotels = pd.read_parquet(DATA / "hotels.parquet")
categories = pd.read_parquet(DATA / "room_categories.parquet")
guests = pd.read_parquet(DATA / "guests.parquet")

In [ ]:
daily_rates = pd.read_parquet(DATA / "daily_rates.parquet")
events = pd.read_parquet(DATA / "calendar_events.parquet")
columns = ["id", "guest_id", "category_id", "checkin", "checkout", "status", "created_at",
           "rate_per_night", "adults", "children", "channel", "commission_pct"]
reservations = pd.read_parquet(DATA / "reservations.parquet", columns=columns)

In [ ]:
reservations.head()

Datumsspalten wandeln wir in echte pandas-Datumswerte um. Danach steht uns `.dt` zur Verfügung.

In [ ]:
reservations["checkin"] = pd.to_datetime(reservations["checkin"])
reservations["checkout"] = pd.to_datetime(reservations["checkout"])
reservations["created_at"] = reservations["created_at"].dt.tz_localize(None)
daily_rates["arrival_date"] = pd.to_datetime(daily_rates["arrival_date"])
events["event_date"] = pd.to_datetime(events["event_date"])

Eine Reservierung kennt nur ihre `category_id`. Das Hotel finden wir über `room_categories`, die Stadt über `hotels`. Das ist derselbe Join wie in SQL, nur mit `merge`.

In [ ]:
rooms = categories.rename(columns={"id": "category_id", "name": "category"})
houses = hotels.rename(columns={"id": "hotel_id"})[["hotel_id", "city_id", "city_type", "bundesland"]]
towns = cities.rename(columns={"id": "city_id", "name": "city"})[["city_id", "city"]]
places = rooms.merge(houses, on="hotel_id").merge(towns, on="city_id")

Für Verkäufe zählen nur bestätigte Aufenthalte, Stornos lassen wir weg.

In [ ]:
stays = reservations.query("status == 'confirmed'").merge(places, on="category_id")
stays["nights"] = (stays["checkout"] - stays["checkin"]).dt.days
stays["persons"] = stays["adults"] + stays["children"]
stays["lead_days"] = (stays["checkin"] - stays["created_at"].dt.normalize()).dt.days

In [ ]:
stays["weekday"] = stays["checkin"].dt.day_of_week
stays["month"] = stays["checkin"].dt.month
stays["year"] = stays["checkin"].dt.year
stays.head()

### Kurz live in die Datenbank

Die Parquet-Dateien sind ein Export. Mit `read_sql` fragen wir die laufende Datenbank direkt ab, und zwar mit dem Nur-Lese-Konto `hotel_student`. Bitte nur kleine Abfragen wie diese: ein Hotel, ein Aggregat. Es gilt ein Timeout von 4 Sekunden. Ohne Verbindung zur Datenbank könnt ihr die beiden Zellen überspringen.

In [ ]:
from app.db_target import resolve_db_url
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
url = resolve_db_url(os.environ, "STUDENT_DATABASE_URL").replace("postgresql://", "postgresql+psycopg://", 1)
engine = create_engine(url)

In [ ]:
sql = text("""
    SELECT c.name, COUNT(*) AS bookings
    FROM room_categories c JOIN reservations r ON r.category_id = c.id
    WHERE c.hotel_id = :hotel_id AND r.status = 'confirmed'
    GROUP BY c.name
""")
pd.read_sql(sql, engine, params={"hotel_id": int(hotels["id"].iloc[0])})

### Wie weit im Voraus wird gebucht?

Der Vorlauf ist die Zeit zwischen Buchung und Anreise. Wie viele Gipfel seht ihr?

In [ ]:
stays["lead_days"].clip(upper=150).plot.hist(bins=50, title="Vorlauf in Tagen")

In [ ]:
bins = [-1, 14, 29, 60, 120, 10_000]
labels = ["0-14", "15-29", "30-60", "61-120", "über 120"]
stays["lead_bucket"] = pd.cut(stays["lead_days"], bins=bins, labels=labels)

In [ ]:
table = stays.pivot_table(index="weekday", columns="lead_bucket", values="id", aggfunc="count", observed=True)
table.index = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]
table.plot.bar(stacked=True, title="Anreisen je Wochentag und Vorlauf")

In [ ]:
stays.groupby("lead_bucket", observed=True)[["nights", "persons"]].mean().round(2)

### Eure Segmentregel

Formuliert eine Regel in einer Zeile mit `np.where` und speichert sie in `stays["segment"]`, mit den Werten `"business"` und `"leisure"`. Danach prüft ihr, ob die beiden Gruppen plausibel aussehen.

In [ ]:
# Hinweis: Welche Spalte von oben trennt die beiden Gruppen am deutlichsten?



In [ ]:
# Hinweis: Vergleicht eure Gruppen mit groupby und agg.



## 2. Kalender-Detektive (10 Min.)

Die Termine in `calendar_events` sind real, ihre Effekte in unseren Daten sind synthetisch.

### Die Osterfalle

Das Management freut sich über den März 2024. Gab es wirklich Wachstum?

In [ ]:
monthly = stays.pivot_table(index="month", columns="year", values="id", aggfunc="count")
monthly[[2023, 2024, 2025]].plot(marker="o", title="Anreisen je Monat")

In [ ]:
events.query("name == 'Ostermontag'")[["event_date", "name"]]

In [ ]:
monthly.loc[[3, 4], [2023, 2024, 2025]]

In [ ]:
monthly.loc[[3, 4], [2023, 2024, 2025]].sum()

### Event-Study

Wir legen alle Ausgaben eines Events übereinander: Tag 0 ist der erste Eventtag, die Kurve zeigt die Anreisen in der Stadt, relativ zu den 14 Tagen davor.

In [ ]:
names = ["Oktoberfest", "Hannover Messe", "Frankfurter Buchmesse", "gamescom", "Karneval"]
selected = events.query("name in @names").astype({"city_id": "int64"})
selected["year"] = selected["event_date"].dt.year
starts = selected.groupby(["name", "city_id", "year"], as_index=False)["event_date"].min()

In [ ]:
city_days = stays.groupby(["city_id", "checkin"]).size().rename("bookings").reset_index()
study = city_days.merge(starts, on="city_id")
study["offset"] = (study["checkin"] - study["event_date"]).dt.days
study = study.query("-14 <= offset <= 21")

In [ ]:
curve = study.pivot_table(index="offset", columns="name", values="bookings", aggfunc="mean")
curve.div(curve.loc[-14:-1].mean()).plot(title="Anreisen rund um Events (relativ)")

### Euro 2024

Vom 14. Juni bis 14. Juli 2024 fand die Europameisterschaft statt. Wir vergleichen Austragungsorte mit den übrigen Städten, jeweils Juni und Juli, relativ zu 2023.

In [ ]:
hosts = events.query("name == 'Euro 2024'")["city_id"].unique()
summer = stays.query("month in [6, 7]").copy()
summer["group"] = np.where(summer["city_id"].isin(hosts), "Austragungsort", "Vergleichsgruppe")

In [ ]:
euro = summer.pivot_table(index="year", columns="group", values="id", aggfunc="count")
euro.div(euro.loc[2023]).plot.bar(title="Anreisen Juni/Juli relativ zu 2023")

### Reserve, falls Zeit bleibt

Wer storniert am häufigsten? Und wann reisen Familien aus welchem Bundesland?

In [ ]:
cancelled = reservations["status"] == "cancelled"
cancelled.groupby(reservations["channel"]).mean().plot.bar(title="Stornoquote je Kanal")

In [ ]:
families = stays.query("children > 0 and year == 2025 and month in [6, 7, 8, 9]")
families = families.merge(guests[["id", "home_state"]], left_on="guest_id", right_on="id", suffixes=("", "_guest"))
families["week"] = families["checkin"].dt.isocalendar().week
families.pivot_table(index="week", columns="home_state", values="id", aggfunc="count").plot(title="Familien je Woche")

## 3. Preisfalle und Umsatzkurve (20 Min.)

Die Hotelleitung fragt: Welcher Preis bringt den höchsten Umsatz? Dafür bauen wir eine Tabelle mit einer Zeile je Kategorie und Anreisetag. `r` ist der Angebotspreis im Verhältnis zum Listenpreis. Tage ohne Verkauf müssen dabei erhalten bleiben, deshalb `how="left"`.

In [ ]:
sales = stays.groupby(["category_id", "checkin"]).size().rename("bookings").reset_index()
sales = sales.rename(columns={"checkin": "arrival_date"})

In [ ]:
daily = daily_rates.merge(places, on="category_id")
daily = daily.merge(sales, how="left", on=["category_id", "arrival_date"])
daily["bookings"] = daily["bookings"].fillna(0)
daily["r"] = (daily["offered_rate"] / daily["base_price"]).round(2)

### Der erste Blick

So sieht das Hotel seine Zahlen im Regelbetrieb. Was würdet ihr der Hotelleitung raten?

In [ ]:
regular_days = daily.query("price_source == 'dynamic'")
regular_days.groupby("r")["bookings"].mean().plot(marker="o", title="Preis gegen Verkäufe")

### Woher kommt diese Kurve?

Glaubt ihr, dass höhere Preise mehr Gäste anlocken? Sucht nach einer anderen Erklärung.

In [ ]:
# Hinweis: Markiert die Tage, an denen in der Stadt ein Event stattfindet.



In [ ]:
# Hinweis: Vergleicht Preis und Verkäufe nach Saison und Event.



In [ ]:
# Hinweis: Zeichnet die Kurve von oben noch einmal, getrennt nach eurer Tagesart.



### Was verkauft wurde und was gewollt war

Sind Verkäufe dasselbe wie Nachfrage? Schaut euch `closed_to_arrival` an.

In [ ]:
# Hinweis: Wie oft kommt closed_to_arrival vor und was passiert dort mit den Verkäufen?



### Eine faire Preiskurve

Welche Tage erlauben eine faire Aussage darüber, wie Gäste auf den Preis reagieren? Schaut euch noch einmal alle Spalten von `daily` an.

In [ ]:
# Hinweis: Filtert mit query auf geeignete Tage und zeichnet Preis gegen Verkäufe.



### Der Umsatzberg

Preis mal Menge: Bei welchem `r` liegt das Maximum?

In [ ]:
# Hinweis: Berechnet aus Preis und Menge eine neue Spalte und sucht ihr Maximum.



### Gilt das für alle Gäste?

Nutzt eure Segmentregel aus Block 1. Haben beide Gruppen denselben besten Preis?

In [ ]:
# Hinweis: Verbindet die einzelnen Aufenthalte mit den geeigneten Tagen.



In [ ]:
# Hinweis: Rechnet die Umsatzkurve je Segment, pivot_table hilft.



### Brutto oder netto?

Buchungsportale kosten Provision. Was bleibt dem Hotel am Ende?

In [ ]:
# Hinweis: Nutzt rate_per_night, nights und commission_pct.



In [ ]:
# Hinweis: Vergleicht die Vertriebskanäle.



## 4. SpiegelMining im Hotel (15 Min.)

David Kriesel hat in seinem Vortrag „SpiegelMining“ aus öffentlich zugänglichen Artikeldaten erstaunlich viel herausgelesen. Geht das auch mit Buchungsdaten? Eure Frage: **Fällt euch in den Aufenthalten etwas auf, das kein Zufall sein kann?**

Damit der Merge klein bleibt, nehmen wir nur Gäste mit mindestens 4 Aufenthalten.

In [ ]:
guest_stays = stays[["guest_id", "hotel_id", "city_id", "checkin"]]
enough_stays = guest_stays.groupby("guest_id")["checkin"].transform("size") >= 4
guest_stays = guest_stays[enough_stays]
len(guest_stays)

### Zählen

Welche Gäste reisen besonders oft am selben Tag im selben Hotel an? Tipp: Eine Tabelle lässt sich mit sich selbst mergen, und `query("a < b")` entfernt doppelte Paare.

In [ ]:
# Hinweis: Merge auf hotel_id und checkin.



In [ ]:
# Hinweis: Zählt mit groupby und size, sortiert absteigend.



### Sagt die Anzahl schon etwas aus?

Schaut euch die Spitzenreiter genau an. Gibt es harmlose Erklärungen? Findet Kennzahlen oder Filter, die solche Fälle aussortieren. `calendar_events` kann dabei helfen.

In [ ]:
# Hinweis: agg kann mehrere Kennzahlen gleichzeitig berechnen.



In [ ]:
# Hinweis: merge mit indicator=True zeigt, welche Zeilen keinen Partner haben.



In [ ]:
# Hinweis: Wendet eure Filter an und sortiert erneut.



### Wer steckt dahinter?

Holt euch Namen und Firmen dazu. Tauchen Personen mehrfach auf?

In [ ]:
# Hinweis: Verbindet euer Ergebnis mit der Tabelle guests.



### Andere Blickwinkel

Bisher hatten wir eine feste Definition von „zusammen“. Variiert sie und schaut, was sich ändert.

In [ ]:
# Hinweis: Probiert andere Merge-Schlüssel oder verschiebt Datumswerte mit pd.Timedelta.



In [ ]:
# Hinweis: Wertet das Ergebnis genauso aus wie zuvor.



## 5. Darf man das? (5 Min.)

Alle Personen in diesen Daten sind frei erfunden. Bei echten Gästen sähe die Lage anders aus:

- **Zweckbindung (Art. 5 Abs. 1 lit. b DSGVO):** Buchungsdaten werden erhoben, um eine Übernachtung abzuwickeln. Dürfen wir daraus Beziehungen zwischen Gästen ableiten? Auch die Datenminimierung (lit. c) gehört in diese Diskussion.
- **Sensible Rückschlüsse (Art. 9 DSGVO):** Aus harmlosen Daten können Hinweise auf Gesundheit, Religion, politische Haltung, Gewerkschaftszugehörigkeit oder sexuelle Orientierung entstehen. Solche Daten sind besonders geschützt.
- **Pseudonymisierung ist keine Anonymisierung:** Unser Export enthält keine E-Mail-Adressen, und selbst eine reine `guest_id` oder ein `card_token` verknüpft weiterhin alle Aufenthalte einer Person. Pseudonymisierte Daten bleiben personenbezogene Daten (Art. 4 Nr. 5, Erwägungsgrund 26 DSGVO).
- **Fehlverdacht:** Wer sehr viele Paare prüft, findet auch dann Auffälligkeiten, wenn gar nichts dahintersteckt.

In [ ]:
guest_count = guests["id"].nunique()
possible_pairs = guest_count * (guest_count - 1) // 2
print(f"{guest_count:,} Gäste ergeben {possible_pairs:,} mögliche Paare.")
print(f"Schon bei einem Treffer auf eine Million Paare wären das {possible_pairs // 1_000_000:,} Verdachtsfälle.")

### Diskussion

1. Wer in einem Hotelkonzern dürfte eine solche Auswertung starten, und wer müsste sie genehmigen?
2. Was würde passieren, wenn ein Treffer falsch ist und die betroffene Person davon erfährt?
3. Welche Spalten würdet ihr vor einem Export entfernen oder vergröbern, und welche Auswertungen gingen dann nicht mehr?
4. Wo verläuft für euch die Grenze zwischen Preisoptimierung und Überwachung?